# EcoGraphRAG — Notebook 1: Embeddings + FAISS Index

**Goal:** Generate nomic-embed-text embeddings for all chunks and build a FAISS index.

**Hardware:** Colab T4 GPU

**Inputs:** `chunks.json` (from laptop chunker)

**Outputs:** `faiss_index.bin` + `chunk_mapping.json` + `embeddings.npy` → saved to Drive

## 1. Setup & Drive Mount

In [ ]:
from google.colab import drive
import os, json, time

drive.mount('/content/drive')

# Project paths on Drive
DRIVE_ROOT = '/content/drive/MyDrive/graphrag_research/'
DRIVE_DATA = DRIVE_ROOT + 'data/'
DRIVE_INDICES = DRIVE_ROOT + 'indices/'

for d in [DRIVE_DATA, DRIVE_INDICES]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted. Project dirs ready.')

In [ ]:
# Install dependencies
!pip install -q sentence-transformers faiss-cpu numpy tqdm

In [ ]:
# Verify GPU
import torch
import numpy as np

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

## 2. Load Chunks

Upload `chunks.json` from your laptop to `DRIVE_DATA`, or run this after the chunker.

In [ ]:
CHUNKS_FILE = DRIVE_DATA + 'chunks.json'

if not os.path.exists(CHUNKS_FILE):
    print('chunks.json not found on Drive!')
    print(f'Please upload chunks.json to: {DRIVE_DATA}')
    print('Run `py -3.13 -m src.chunker` on your laptop first.')
else:
    with open(CHUNKS_FILE, 'r') as f:
        chunks = json.load(f)
    print(f'Loaded {len(chunks)} chunks')
    print(f'Sample: {chunks[0]["chunk_id"]} — {chunks[0]["text"][:100]}...')

# Build chunk lookup (chunk_id -> chunk dict)
chunk_lookup = {c['chunk_id']: c for c in chunks}
# Build chunk mapping (index position -> chunk_id) — always from current chunks
chunk_mapping = [c['chunk_id'] for c in chunks]
print(f'Chunk mapping: {len(chunk_mapping)} entries')
print(f'Chunk lookup: {len(chunk_lookup)} entries')

## 3. Generate Embeddings with nomic-embed-text

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = 'nomic-ai/nomic-embed-text-v1'
EMBEDDING_DIM = 768
BATCH_SIZE = 64  # Adjust if OOM

# Check for existing embeddings
EMBEDDINGS_FILE = DRIVE_INDICES + 'embeddings.npy'

if os.path.exists(EMBEDDINGS_FILE):
    embeddings = np.load(EMBEDDINGS_FILE)
    print(f'Loaded existing embeddings: shape={embeddings.shape}')
    # Verify they match our chunks
    if embeddings.shape[0] != len(chunks):
        print(f'WARNING: embeddings ({embeddings.shape[0]}) != chunks ({len(chunks)}). Rebuilding...')
        os.remove(EMBEDDINGS_FILE)
        # Fall through to rebuild
    else:
        print('Embeddings match chunks count. Skipping encoding.')

if not os.path.exists(EMBEDDINGS_FILE):
    print(f'Loading model: {EMBEDDING_MODEL}')
    embed_model = SentenceTransformer(EMBEDDING_MODEL, trust_remote_code=True)
    embed_model = embed_model.to('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Model loaded on {embed_model.device}')

    # nomic-embed-text requires 'search_document: ' prefix for documents
    texts = ['search_document: ' + c['text'] for c in chunks]

    print(f'Encoding {len(texts)} chunks...')
    start = time.time()

    embeddings = embed_model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,  # L2 normalize for cosine similarity
    )

    elapsed = time.time() - start
    print(f'Encoding done in {elapsed:.1f}s ({len(texts)/elapsed:.0f} chunks/sec)')
    print(f'Embeddings shape: {embeddings.shape}')

    # Save embeddings
    np.save(EMBEDDINGS_FILE, embeddings)
    size_mb = os.path.getsize(EMBEDDINGS_FILE) / (1024**2)
    print(f'Saved embeddings to {EMBEDDINGS_FILE} ({size_mb:.1f} MB)')

    # Free model from GPU (we'll reload a lighter version for queries)
    del embed_model
    torch.cuda.empty_cache()

## 4. Build FAISS Index

In [ ]:
import faiss

FAISS_INDEX_FILE = DRIVE_INDICES + 'faiss_index.bin'
CHUNK_MAPPING_FILE = DRIVE_INDICES + 'chunk_mapping.json'
FAISS_TOP_K = 5

# Always rebuild if embeddings count doesn't match index
rebuild = False

if os.path.exists(FAISS_INDEX_FILE):
    index = faiss.read_index(FAISS_INDEX_FILE)
    if index.ntotal != len(chunks):
        print(f'WARNING: FAISS index ({index.ntotal}) != chunks ({len(chunks)}). Rebuilding...')
        rebuild = True
    else:
        print(f'FAISS index loaded: {index.ntotal} vectors')
else:
    rebuild = True

if rebuild:
    print(f'Building FAISS index with {len(embeddings)} vectors (dim={embeddings.shape[1]})')
    start = time.time()

    # Use IndexFlatIP for cosine similarity (embeddings are L2-normalized)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings.astype(np.float32))

    elapsed = time.time() - start
    print(f'Index built in {elapsed:.2f}s — {index.ntotal} vectors')

    # Save FAISS index
    faiss.write_index(index, FAISS_INDEX_FILE)
    size_mb = os.path.getsize(FAISS_INDEX_FILE) / (1024**2)
    print(f'Saved FAISS index to {FAISS_INDEX_FILE} ({size_mb:.1f} MB)')

# Always save chunk mapping from current chunks (keeps it in sync)
with open(CHUNK_MAPPING_FILE, 'w') as f:
    json.dump(chunk_mapping, f)
print(f'Saved chunk mapping ({len(chunk_mapping)} entries)')

## 5. Retrieval Test

In [ ]:
# Load embedding model for queries (keep in memory for reuse)
query_model = SentenceTransformer(EMBEDDING_MODEL, trust_remote_code=True)

def retrieve(query, top_k=FAISS_TOP_K):
    """Retrieve top-k chunks for a query."""
    # nomic-embed-text requires 'search_query: ' prefix for queries
    q_emb = query_model.encode(
        ['search_query: ' + query],
        normalize_embeddings=True,
    ).astype(np.float32)

    scores, indices = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        cid = chunk_mapping[idx]
        if cid not in chunk_lookup:
            print(f'  WARNING: chunk_id "{cid}" not found in chunks — skipping')
            continue
        chunk = chunk_lookup[cid]
        results.append({
            'chunk_id': cid,
            'score': float(score),
            'title': chunk['title'],
            'text': chunk['text'][:200],
        })
    return results

# Test queries
test_questions = [
    'Were Scott Derrickson and Ed Wood of the same nationality?',
    'Which magazine was started first Arthur\'s Magazine or First for Women?',
]

for test_q in test_questions:
    print(f'Query: {test_q}\n')
    for r in retrieve(test_q):
        print(f"  [{r['score']:.3f}] {r['chunk_id']} | {r['title']}")
        print(f"          {r['text'][:120]}...\n")

## 6. Cost Metrics

In [ ]:
# Record indexing cost for Table 2
if torch.cuda.is_available():
    gpu_mem = torch.cuda.max_memory_allocated() / (1024**2)
else:
    gpu_mem = 0

index_metrics = {
    'num_chunks': len(chunks),
    'embedding_dim': EMBEDDING_DIM,
    'embedding_model': EMBEDDING_MODEL,
    'peak_gpu_mb': round(gpu_mem, 2),
    'faiss_index_size_mb': round(os.path.getsize(FAISS_INDEX_FILE) / (1024**2), 2),
}

metrics_file = DRIVE_INDICES + 'index_metrics.json'
with open(metrics_file, 'w') as f:
    json.dump(index_metrics, f, indent=2)

print('Indexing metrics:')
for k, v in index_metrics.items():
    print(f'  {k}: {v}')

print(f'\n✅ Embeddings + FAISS index ready on Drive!')